# Real-World Experiment Pipeline

Pipeline:
1. Capture RGBD from RealSense → point cloud
2. Flatten table plane to Z=0, crop workspace, remove table-color points
3. DBSCAN clustering → one point cloud per tool
4. **Match each cluster to hammer / spatula / L-ruler, estimate yaw** ← new
5. Reconstruct the scene in the RAI simulation at the estimated poses

In [1]:
import colorsys
import math

import numpy as np
import open3d as o3d
from sklearn.decomposition import PCA
from scipy.spatial import cKDTree

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 1 — Point cloud capture

Run this cell with the RealSense plugged in, **or** skip it and load a saved `.ply` in the next section.

In [2]:
import pyrealsense2 as rs
import cv2


def get_realsense_rgbd():
    pipeline = rs.pipeline()
    config = rs.config()
    config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
    config.enable_stream(rs.stream.depth, 640, 480, rs.format.z16, 30)
    profile = pipeline.start(config)
    align = rs.align(rs.stream.color)
    try:
        for _ in range(30):          # let auto-exposure settle
            pipeline.wait_for_frames()
        frames = pipeline.wait_for_frames()
        aligned = align.process(frames)
        color_frame = aligned.get_color_frame()
        depth_frame = aligned.get_depth_frame()
        if not color_frame or not depth_frame:
            raise RuntimeError("Could not get frames.")
        color_image = np.asanyarray(color_frame.get_data())
        depth_image = np.asanyarray(depth_frame.get_data())
        intrinsics = color_frame.profile.as_video_stream_profile().intrinsics
        return color_image, depth_image, intrinsics
    finally:
        pipeline.stop()


def create_point_cloud(color_image, depth_image, intrinsics):
    color_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
    rgbd = o3d.geometry.RGBDImage.create_from_color_and_depth(
        o3d.geometry.Image(color_rgb),
        o3d.geometry.Image(depth_image),
        depth_scale=1000.0,
        depth_trunc=2.0,
        convert_rgb_to_intensity=False,
    )
    cam = o3d.camera.PinholeCameraIntrinsic(
        intrinsics.width, intrinsics.height,
        intrinsics.fx, intrinsics.fy,
        intrinsics.ppx, intrinsics.ppy,
    )
    pcd = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd, cam)
    # Flip to standard camera frame
    pcd.transform([[1,0,0,0],[0,-1,0,0],[0,0,-1,0],[0,0,0,1]])
    return pcd


color_image, depth_image, intrinsics = get_realsense_rgbd()
raw_pcd = create_point_cloud(color_image, depth_image, intrinsics)
o3d.visualization.draw_geometries([raw_pcd])
o3d.io.write_point_cloud("realsense_point_cloud.ply", raw_pcd)
print("Saved realsense_point_cloud.ply")

Saved realsense_point_cloud.ply


## 2 — Preprocessing: flatten table, crop, remove table colour

In [3]:
# ── Load saved cloud ────────────────────────────────────────────────────────
raw_pcd = o3d.io.read_point_cloud("realsense_point_cloud.ply")
raw_pcd = raw_pcd.voxel_down_sample(voxel_size=0.005)


# ── Rotation helpers ────────────────────────────────────────────────────────
def rotation_matrix_from_vectors(a, b):
    """Rotation matrix that rotates unit vector a onto unit vector b."""
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    v = np.cross(a, b)
    c = np.dot(a, b)
    if np.isclose(c, 1.0):
        return np.eye(3)
    if np.isclose(c, -1.0):
        ax = np.array([1.0, 0.0, 0.0])
        if abs(a[0]) > 0.9:
            ax = np.array([0.0, 1.0, 0.0])
        v = np.cross(a, ax)
        v /= np.linalg.norm(v)
        vx = np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
        return np.eye(3) + 2 * vx @ vx
    s = np.linalg.norm(v)
    vx = np.array([[0,-v[2],v[1]],[v[2],0,-v[0]],[-v[1],v[0],0]])
    return np.eye(3) + vx + vx @ vx * ((1 - c) / (s ** 2))


def flatten_table_to_z0(pcd, distance_threshold=0.01):
    """RANSAC plane detection → rotate table normal to +Z, translate table to z=0."""
    plane_model, inliers = pcd.segment_plane(
        distance_threshold=distance_threshold, ransac_n=3, num_iterations=2000
    )
    a, b, c, d = plane_model
    normal = np.array([a, b, c], dtype=float)
    normal /= np.linalg.norm(normal)
    if np.dot(normal, [0, 0, 1]) < 0:
        normal = -normal
    R = rotation_matrix_from_vectors(normal, np.array([0.0, 0.0, 1.0]))
    pts = np.asarray(pcd.points)
    cols = np.asarray(pcd.colors)
    pts_rot = (R @ pts.T).T
    table_z = np.median(pts_rot[inliers, 2])
    pts_rot[:, 2] -= table_z
    # 180° around Z so the scene faces the robot (+Y forward in sim)
    Rz180 = np.array([[-1,0,0],[0,-1,0],[0,0,1]])
    pts_rot = (Rz180 @ pts_rot.T).T
    pcd_out = o3d.geometry.PointCloud()
    pcd_out.points = o3d.utility.Vector3dVector(pts_rot)
    if len(cols) == len(pts):
        pcd_out.colors = o3d.utility.Vector3dVector(cols)
    print(f"Plane normal: {normal}  |  table inliers: {len(inliers)}")
    return pcd_out, R


def crop_pcd(pcd, x_min, x_max, y_min, y_max, z_min, z_max):
    pts = np.asarray(pcd.points)
    mask = (
        (pts[:,0] >= x_min) & (pts[:,0] <= x_max) &
        (pts[:,1] >= y_min) & (pts[:,1] <= y_max) &
        (pts[:,2] >= z_min) & (pts[:,2] <= z_max)
    )
    return pcd.select_by_index(np.where(mask)[0])


import numpy as np
import colorsys


def remove_table_colour(pcd, sat_thresh=0.20, val_thresh=0.80):
    """
    Drop mostly white / very light grey table points.

    White in HSV:
    - low saturation
    - high value
    """
    cols = np.asarray(pcd.colors)

    hsv = np.array([colorsys.rgb_to_hsv(r, g, b) for r, g, b in cols])

    is_white_like = (hsv[:, 1] < sat_thresh) & (hsv[:, 2] > val_thresh)

    keep = ~is_white_like
    return pcd.select_by_index(np.where(keep)[0])


# ── Run preprocessing ───────────────────────────────────────────────────────
pcd_flat, _ = flatten_table_to_z0(raw_pcd)
pts = np.asarray(pcd_flat.points)
print(f"Object cloud: {len(pts)} points")
print(f"x [{pts[:,0].min():.3f}, {pts[:,0].max():.3f}]")
print(f"y [{pts[:,1].min():.3f}, {pts[:,1].max():.3f}]")
print(f"z [{pts[:,2].min():.3f}, {pts[:,2].max():.3f}]")

# # Adjust these bounds to your workspace
pcd_cropped = crop_pcd(pcd_flat, x_min=-1.087, x_max=0.92,
                                  y_min=-0.93, y_max=-0.44,
                                  z_min=0.0020, z_max=0.353)

objects_pcd = remove_table_colour(pcd_cropped,
    sat_thresh=0.20,
    val_thresh=0.55)

# pts = np.asarray(objects_pcd.points)
# print(f"Object cloud: {len(pts)} points")
# print(f"x [{pts[:,0].min():.3f}, {pts[:,0].max():.3f}]")
# print(f"y [{pts[:,1].min():.3f}, {pts[:,1].max():.3f}]")
# print(f"z [{pts[:,2].min():.3f}, {pts[:,2].max():.3f}]")

axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.15)
o3d.visualization.draw_geometries([objects_pcd, axis])

Plane normal: [0.00200346 0.94628515 0.32332708]  |  table inliers: 17140
Object cloud: 61835 points
x [-1.061, 1.044]
y [-2.145, -0.483]
z [-0.028, 0.550]


## 3 — DBSCAN clustering

In [4]:
import numpy as np
import open3d as o3d


def cluster_objects_exact_n(
    pcd,
    expected_n=4,
    voxel_size=0.004,
    eps=0.025,
    min_points=20,
    min_cluster_points=80,
    keep_largest=True,
    visualize=True,
):
    """
    DBSCAN clustering, but returns exactly expected_n clusters if possible.

    Important:
    - DBSCAN itself cannot be forced to produce exactly 4 clusters.
    - This function runs DBSCAN, removes tiny clusters, and if more than 4 remain,
      keeps the most useful 4 clusters.
    """

    # 1. Downsample
    pcd_down = pcd.voxel_down_sample(voxel_size)

    # 2. Remove outliers FROM THE DOWNSAMPLED CLOUD
    pcd_clean, _ = pcd_down.remove_statistical_outlier(
        nb_neighbors=20,
        std_ratio=2.0
    )

    # 3. DBSCAN
    labels = np.array(
        pcd_clean.cluster_dbscan(
            eps=eps,
            min_points=min_points,
            print_progress=True
        )
    )

    clusters = []

    for lbl in sorted(set(labels)):
        if lbl == -1:
            continue

        idx = np.where(labels == lbl)[0]

        if len(idx) < min_cluster_points:
            continue

        c = pcd_clean.select_by_index(idx)
        clusters.append(c)

    print(f"\nDBSCAN found {len(clusters)} valid clusters before enforcing expected_n={expected_n}")

    # 4. If too many clusters, keep exactly expected_n
    if len(clusters) > expected_n:
        if keep_largest:
            clusters = sorted(
                clusters,
                key=lambda c: len(c.points),
                reverse=True
            )[:expected_n]
        else:
            clusters = clusters[:expected_n]

        print(f"Kept {expected_n} clusters.")

    # 5. If too few clusters, warn clearly
    if len(clusters) < expected_n:
        print(
            f"WARNING: only {len(clusters)} clusters found, "
            f"but expected {expected_n}. Try increasing eps or decreasing min_points."
        )

    # 6. Print and visualize
    vis = []

    for i, c in enumerate(clusters):
        c_vis = o3d.geometry.PointCloud(c)
        c_vis.paint_uniform_color(np.random.rand(3))

        aabb = c.get_axis_aligned_bounding_box()
        aabb.color = [1, 0, 0]

        center = np.asarray(aabb.get_center())
        extent = np.asarray(aabb.get_extent())

        print(
            f"Cluster {i}: {len(c.points)} pts | "
            f"center={center.round(3)} | "
            f"extent={extent.round(3)}"
        )

        vis += [c_vis, aabb]

    if visualize:
        axis = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.15)
        o3d.visualization.draw_geometries(vis + [axis])

    return clusters

clusters = cluster_objects_exact_n(
    objects_pcd,
    expected_n=4,
    voxel_size=0.004,
    eps=0.025,
    min_points=20,
    min_cluster_points=80,
    keep_largest=True,
    visualize=True,
)


DBSCAN found 4 valid clusters before enforcing expected_n=4
Cluster 0: 6447 pts | center=[ 0.216 -0.678  0.177] | extent=[0.221 0.284 0.35 ]
Cluster 1: 431 pts | center=[ 0.032 -0.656  0.013] | extent=[0.095 0.212 0.02 ]
Cluster 2: 591 pts | center=[-0.144 -0.569  0.013] | extent=[0.169 0.173 0.021]
Cluster 3: 393 pts | center=[-0.286 -0.71   0.011] | extent=[0.126 0.204 0.017]
Precompute neighbors.[========================================] 100%


## 5 — Reconstruct scene in simulation

We place each matched tool in RAI at the estimated position and yaw.

> **Coordinate frame note**: the `position` above is expressed in the *camera-aligned,
> table-flattened* frame.  If your camera is not directly above the robot workspace you
> may need to apply an additional rigid transform here to map into the simulation frame.
> Set `CAM_TO_SIM` below to do that.

In [5]:
import numpy as np

def quat_from_yaw(yaw):
    """
    Convert yaw angle around z-axis to RAI quaternion format.
    RAI quaternion format: [w, x, y, z]
    """
    return [
        float(np.cos(yaw / 2.0)),
        0.0,
        0.0,
        float(np.sin(yaw / 2.0)),
    ]

print("quat_from_yaw defined.")

quat_from_yaw defined.


In [6]:
import robotic as ry
import WayTu_RAI.model_utils as mutils
from WayTu_RAI.CameraRAI import CameraRAI




class RealWorldToolPlacer:
    """
    Places the default simulation tools at the real-world estimated poses.
    Mirrors DefaultToolGenerator from the retired notebook but takes its
    parameters from the matcher output rather than random values.
    """

    # Tool geometry (matches PrimitiveTools.py)
    HANDLE_SHAPES = {
        'hammer':  [0.035, 0.25, 0.035, 0.008],
        'spatula': [0.025, 0.25, 0.025, 0.005],
        'L-ruler': [0.04,  0.20, 0.02,  0.005],
    }
    HEAD_SHAPES = {
        'hammer':  [0.07,  0.04,  0.04,  0.010],
        'spatula': [0.08,  0.08,  0.01,  0.005],
        'L-ruler': [0.15,  0.04,  0.02,  0.005],
    }
    HEAD_POSITIONS = {
        'hammer':  [0.0,  0.12, 0.0],
        'spatula': [0.0,  0.09, 0.0],
        'L-ruler': [0.06, 0.10, 0.0],   # side=+1; flip X for side=-1
    }
    HEAD_YAW_EXTRA = {
        'hammer':  0.0,
        'spatula': 0.0,
        'L-ruler': math.pi / 2,
    }
    HANDLE_COLOURS = {
        'hammer':  [0.55, 0.27, 0.07],
        'spatula': [0.15, 0.15, 0.15],
        'L-ruler': [0.65, 0.65, 0.70],
    }
    HEAD_COLOURS = {
        'hammer':  [0.70, 0.70, 0.75],
        'spatula': [0.80, 0.80, 0.80],
        'L-ruler': [0.65, 0.65, 0.70],
    }

    def __init__(self, C):
        self.C = C

    def place(self, tool_type, parent='table'):
        """
        Add one tool to the RAI config.

        Parameters
        ----------
        tool_type : 'hammer' | 'spatula' | 'L-ruler'
        position  : [x, y, z]  (z is height above table surface)
        yaw       : float (radians)
        parent    : RAI frame name for the table
        """



        mutils.add_shape(
            C=self.C,
            frame_name=f"{tool_type}-base",
            parent=parent,
            shape=self.HANDLE_SHAPES[tool_type],
            relative_position=[0.0, 0.3, 0.06],
            relative_quaternion=[1, 0, 0, 0],
            joint=True,
            color=self.HANDLE_COLOURS[tool_type],
        )
        mutils.add_shape(
            C=self.C,
            frame_name=f"{tool_type}-head",
            parent=f"{tool_type}-base",
            shape=self.HEAD_SHAPES[tool_type],
            relative_position=self.HEAD_POSITIONS[tool_type],
            relative_quaternion=[1, 0, 0, 0],
            color=self.HEAD_COLOURS[tool_type],
        )



print("RealWorldToolPlacer defined.")

RealWorldToolPlacer defined.


In [ ]:
class RealWorldPlatformPlacer:
    """
    Places task platforms at an estimated real-world pose.

    Supported tasks:
        - lifting
        - minigolf
        - hammering
    """

    def __init__(self, C):
        self.C = C

    def place(
        self,
        task,
        position,
        yaw,
        platform_height=0.4,
        parent="table",
        prefix=None,
    ):
        """
        task: "lifting", "minigolf", or "hammering"
        position: center of platform in RAI/world frame
        yaw: platform yaw in radians
        """

        if task == "lifting":
            if prefix is None:
                prefix = "real-lifting"

            return self.place_lifting(
                position=position,
                yaw=yaw,
                platform_height=platform_height,
                parent=parent,
                prefix=prefix,
            )

        elif task == "minigolf":
            if prefix is None:
                prefix = "real-minigolf"

            return self.place_minigolf(
                position=position,
                yaw=yaw,
                parent=parent,
                prefix=prefix,
            )

        elif task == "hammering":
            if prefix is None:
                prefix = "real-hammering"

            return self.place_hammering(
                position=position,
                yaw=yaw,
                parent=parent,
                prefix=prefix,
            )

        else:
            raise ValueError(f"Unknown task: {task}")

    def place_lifting(
        self,
        position,
        yaw,
        platform_height=0.4,
        parent="table",
        prefix="real-lifting",
    ):
        """
        Places lifting platform.

        Important frame names:
            real-lifting-platform
            real-lifting-back-leg1
            real-lifting-back-joint
            real-lifting-back-leg2
            real-lifting-front-leg1
            real-lifting-front-joint
            real-lifting-front-leg2
            real-lifting-lifting-obj
        """

        plate_shape = [0.2, 0.4, 0.04, 0.005]
        leg_shape = [0.04, 0.04, platform_height, 0.005]
        joint_shape = [0.06, 0.04, 0.04, 0.005]
        obj_shape = [0.03, 0.5, 0.03, 0.005]

        quat = quat_from_yaw(yaw)

        platform_name = f"{prefix}-platform"

        mutils.add_shape(
            C=self.C,
            frame_name=platform_name,
            parent=parent,
            shape=plate_shape,
            relative_position=position,
            relative_quaternion=quat,
            color=[0.1],
            joint=True,
            mass=50.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-back-leg1",
            parent=platform_name,
            shape=leg_shape,
            color=[0.1],
            relative_position=[-0.04, -0.2, platform_height / 2.0],
            mass=5.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-back-joint",
            parent=f"{prefix}-back-leg1",
            shape=joint_shape,
            color=[0.1],
            relative_position=[0.04, 0.0, platform_height / 2.0 - 0.045],
            mass=5.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-back-leg2",
            parent=f"{prefix}-back-joint",
            shape=leg_shape,
            color=[0.1],
            relative_position=[0.04, 0.0, -platform_height / 2.0 + 0.045],
            mass=5.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-front-leg1",
            parent=platform_name,
            shape=leg_shape,
            color=[0.1],
            relative_position=[-0.04, 0.2, platform_height / 2.0],
            mass=5.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-front-joint",
            parent=f"{prefix}-front-leg1",
            shape=joint_shape,
            color=[0.1],
            relative_position=[0.04, 0.0, platform_height / 2.0 - 0.045],
            mass=5.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-front-leg2",
            parent=f"{prefix}-front-joint",
            shape=leg_shape,
            color=[0.1],
            relative_position=[0.04, 0.0, -platform_height / 2.0 + 0.045],
            mass=5.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-lifting-obj",
            parent=f"{prefix}-back-joint",
            shape=obj_shape,
            color=[0.0, 0.0, 1.0],
            relative_position=[0.0, 0.2, 0.04],
            joint=True,
            mass=0.00001,
        )

        return platform_name

    def place_minigolf(
        self,
        position,
        yaw,
        parent="table",
        prefix="real-minigolf",
    ):
        """
        Places minigolf platform.

        Important frame names:
            real-minigolf-platform
            real-minigolf-main-area
            real-minigolf-left-area
            real-minigolf-right-area
            real-minigolf-end-area
            real-minigolf-minigolf-obj
        """

        minigolf_platform_shape = [0.26, 0.4, 0.01, 0.005]
        main_area_shape = [0.26, 0.2, 0.1, 0.005]
        left_area_shape = [0.06, 0.1, 0.1, 0.005]
        right_area_shape = [0.06, 0.1, 0.1, 0.005]
        end_area_shape = [0.26, 0.1, 0.1, 0.005]
        minigolf_obj_shape = [0.05, 0.05, 0.05, 0.02]

        quat = quat_from_yaw(yaw)

        platform_name = f"{prefix}-platform"

        mutils.add_shape(
            C=self.C,
            frame_name=platform_name,
            parent=parent,
            shape=minigolf_platform_shape,
            relative_position=position,
            relative_quaternion=quat,
            color=[1.0, 1.0, 1.0],
            joint=True,
            mass=50.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-main-area",
            parent=platform_name,
            shape=main_area_shape,
            color=[0.1],
            relative_position=[0.0, 0.1, 0.0505],
            mass=40.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-left-area",
            parent=f"{prefix}-main-area",
            shape=left_area_shape,
            color=[0.1],
            relative_position=[0.1, -0.15, 0.0],
            mass=20.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-right-area",
            parent=f"{prefix}-main-area",
            shape=right_area_shape,
            color=[0.1],
            relative_position=[-0.1, -0.15, 0.0],
            mass=20.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-end-area",
            parent=f"{prefix}-main-area",
            shape=end_area_shape,
            color=[0.1],
            relative_position=[0.0, -0.25, 0.0],
            mass=30.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-minigolf-obj",
            parent=platform_name,
            shape=minigolf_obj_shape,
            color=[0.0, 0.0, 1.0],
            relative_position=[0.0, 0.1, 0.085],
            joint=True,
            mass=0.00001,
        )

        return platform_name

    def place_hammering(
        self,
        position,
        yaw,
        parent="table",
        prefix="real-hammering",
    ):
        """
        Places hammering platform using HammeringEnvironment structure.

        Important frame names:
            real-hammering-platform
            real-hammering-main-area
            real-hammering-left-area
            real-hammering-right-area
            real-hammering-top-area
            real-hammering-nail-base
            real-hammering-hammering-obj
        """

        hammering_platform_shape = [0.20, 0.25, 0.01, 0.001]
        main_area_shape = [0.20, 0.15, 0.25, 0.001]
        side_area_shape = [0.09, 0.15, 0.02, 0.001]
        top_area_shape = [0.20, 0.15, 0.1, 0.001]

        nail_base_shape = [0.018, 0.15, 0.018, 0.001]
        nail_head_shape = [0.04, 0.01, 0.04, 0.001]

        quat = quat_from_yaw(yaw)

        platform_name = f"{prefix}-platform"

        mutils.add_shape(
            C=self.C,
            frame_name=platform_name,
            parent=parent,
            shape=hammering_platform_shape,
            relative_position=position,
            relative_quaternion=quat,
            color=(0.76, 0.60, 0.42),
            joint=True,
            mass=1000.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-main-area",
            parent=platform_name,
            shape=main_area_shape,
            color=(0.76, 0.60, 0.42),
            relative_position=[0.0, -0.05, 0.13],
            mass=40.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-left-area",
            parent=f"{prefix}-main-area",
            shape=side_area_shape,
            color=(0.76, 0.60, 0.42),
            relative_position=[-0.055, 0.0, 0.135],
            mass=20.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-right-area",
            parent=f"{prefix}-main-area",
            shape=side_area_shape,
            color=(0.76, 0.60, 0.42),
            relative_position=[0.055, 0.0, 0.135],
            mass=20.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-top-area",
            parent=f"{prefix}-main-area",
            shape=top_area_shape,
            color=(0.76, 0.60, 0.42),
            relative_position=[0.0, 0.0, 0.195],
            mass=20.0,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-nail-base",
            parent=platform_name,
            shape=nail_base_shape,
            color=[0.3, 0.5, 0.8],
            joint=True,
            mass=0.001,
        )

        mutils.add_shape(
            C=self.C,
            frame_name=f"{prefix}-hammering-obj",
            parent=f"{prefix}-nail-base",
            shape=nail_head_shape,
            color=[0.3, 0.5, 0.8],
            relative_position=[0.0, 0.08, 0.0],
            mass=0.001,
        )
        obj = self.C.getFrame("nail-base")
        platform = self.C.getFrame("hammering-platform").getPosition()
        obj.setPosition(platform + [0.0, 0.0, 0.265])

        return platform_name


# Backward compatibility:
# If the rest of your notebook still calls RealWorldLiftingPlatformPlacer(C),
# it will still work.
RealWorldLiftingPlatformPlacer = RealWorldPlatformPlacer

In [8]:
def add_robot_reference_marker(C, position, yaw=0.0, parent="table", name="robot-reference"):
    """
    Debug marker for the detected robot/reference cluster.
    This is not the real robot. It is only an anchor marker.
    """

    mutils.add_shape(
        C=C,
        frame_name=name,
        parent=parent,
        shape=[0.06, 0.06, 0.02, 0.005],
        relative_position=position,
        relative_quaternion=quat_from_yaw(yaw),
        color=[1.0, 0.0, 0.0],
        joint=False,
        mass=0.01,
    )

    return name

In [ ]:
# ============================================================
# SIMULATION TEMPLATE POINT CLOUDS FOR MATCHING
# ============================================================

TASK = "hammering"   # "lifting", "minigolf", or "hammering"

cameras = ["camera1", "camera2", "camera3"]

simulation_point_clouds = {}


def add_lifting_template_platform(C):
    """
    Simple lifting platform template for matching.
    Uses the same naming as the matching object: lifting-platform.
    """

    plate_shape = [0.2, 0.4, 0.04, 0.005]
    obj_shape = [0.03, 0.5, 0.03, 0.005]

    mutils.add_shape(
        C=C,
        frame_name="lifting-platform",
        parent="table",
        shape=plate_shape,
        relative_position=[0.0, 0.3, 0.06],
        relative_quaternion=[1, 0, 0, 0],
        color=[0.1],
        joint=True,
        mass=50.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="lifting-obj",
        parent="lifting-platform",
        shape=obj_shape,
        relative_position=[0.0, 0.0, 0.25],
        relative_quaternion=[1, 0, 0, 0],
        color=[0.0, 0.0, 1.0],
        joint=True,
        mass=0.00001,
    )


def add_minigolf_template_platform(C):
    """
    Minigolf platform template for matching.
    Uses the original MinigolfEnvironment structure.
    """

    minigolf_platform_shape = [0.26, 0.4, 0.01, 0.005]
    main_area_shape = [0.26, 0.2, 0.1, 0.005]
    left_area_shape = [0.06, 0.1, 0.1, 0.005]
    right_area_shape = [0.06, 0.1, 0.1, 0.005]
    end_area_shape = [0.26, 0.1, 0.1, 0.005]
    minigolf_obj_shape = [0.05, 0.05, 0.05, 0.02]

    mutils.add_shape(
        C=C,
        frame_name="minigolf-platform",
        parent="table",
        shape=minigolf_platform_shape,
        relative_position=[0.0, 0.3, 0.01],
        relative_quaternion=[1, 0, 0, 0],
        color=[1.0, 1.0, 1.0],
        joint=True,
        mass=50.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="main-area",
        parent="minigolf-platform",
        shape=main_area_shape,
        color=[0.1],
        relative_position=[0.0, 0.1, 0.0505],
        mass=40.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="left-area",
        parent="main-area",
        shape=left_area_shape,
        color=[0.1],
        relative_position=[0.1, -0.15, 0.0],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="right-area",
        parent="main-area",
        shape=right_area_shape,
        color=[0.1],
        relative_position=[-0.1, -0.15, 0.0],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="end-area",
        parent="main-area",
        shape=end_area_shape,
        color=[0.1],
        relative_position=[0.0, -0.25, 0.0],
        mass=30.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="minigolf-obj",
        parent="minigolf-platform",
        shape=minigolf_obj_shape,
        color=[0.0, 0.0, 1.0],
        relative_position=[0.0, 0.1, 0.085],
        joint=True,
        mass=0.00001,
    )


def add_hammering_template_platform(C):
    """
    Hammering platform template for matching.
    Uses the original HammeringEnvironment structure.

    Object name for matching:
        hammering-platform
    """

    hammering_platform_shape = [0.20, 0.25, 0.01, 0.001]
    main_area_shape = [0.20, 0.15, 0.25, 0.001]
    side_area_shape = [0.09, 0.15, 0.02, 0.001]
    top_area_shape = [0.20, 0.15, 0.1, 0.001]

    nail_base_shape = [0.018, 0.15, 0.018, 0.001]
    nail_head_shape = [0.04, 0.01, 0.04, 0.001]

    mutils.add_shape(
        C=C,
        frame_name="hammering-platform",
        parent="table",
        shape=hammering_platform_shape,
        relative_position=[0.0, 0.3, 0.01],
        relative_quaternion=[1, 0, 0, 0],
        color=(0.76, 0.60, 0.42),
        joint=True,
        mass=1000.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="main-area",
        parent="hammering-platform",
        shape=main_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[0.0, -0.05, 0.13],
        mass=40.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="left-area",
        parent="main-area",
        shape=side_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[-0.055, 0.0, 0.135],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="right-area",
        parent="main-area",
        shape=side_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[0.055, 0.0, 0.135],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="top-area",
        parent="main-area",
        shape=top_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[0.0, 0.0, 0.195],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="nail-base",
        parent="hammering-platform",
        shape=nail_base_shape,
        color=[0.3, 0.5, 0.8],
        joint=True,
        mass=0.001,
    )

    mutils.add_shape(
        C=C,
        frame_name="hammering-obj",
        parent="nail-base",
        shape=nail_head_shape,
        color=[0.3, 0.5, 0.8],
        relative_position=[0.0, 0.08, 0.0],
        mass=0.001,
    )
    obj = C.getFrame("nail-base")
    platform = C.getFrame("hammering-platform").getPosition()
    obj.setPosition(platform + [0.0, 0.0, 0.265])


if TASK == "lifting":
    sim_objects = ["spatula", "L-ruler", "hammer", "lifting-platform"]

elif TASK == "minigolf":
    sim_objects = ["spatula", "L-ruler", "hammer", "minigolf-platform"]

elif TASK == "hammering":
    sim_objects = ["spatula", "L-ruler", "hammer", "hammering-platform"]

else:
    raise ValueError(f"Unknown TASK: {TASK}")


for obj_name in sim_objects:
    C = ry.Config()
    C.addFile("./rai-robotModels/scenarios/pandaSingle.g")

    if obj_name in ["spatula", "L-ruler", "hammer"]:
        placer = RealWorldToolPlacer(C)
        placer.place(tool_type=obj_name)

    elif obj_name == "lifting-platform":
        add_lifting_template_platform(C)

    elif obj_name == "minigolf-platform":
        add_minigolf_template_platform(C)

    elif obj_name == "hammering-platform":
        add_hammering_template_platform(C)

    else:
        raise ValueError(f"Unknown simulation object: {obj_name}")

    camera = CameraRAI({"cameras": cameras}, C)
    pcl, rgb, _ = camera.getPointCloud()
    pcl, rgb = camera.cleanPointClouds(pcl, rgb)

    camera.draw_in_simulation(
        pcl,
        name=f"{obj_name}_template_pcl",
        color=[1.0, 0.0, 0.0],
        rgb=rgb,
    )

    simulation_point_clouds[obj_name] = pcl

print("Simulation templates created:")
for name, pcl in simulation_point_clouds.items():
    print(name, np.asarray(pcl).shape)

{'name': 'table', 'ID': 1, 'rel': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0], 'shape': 'ssBox', 'size': [2.5, 2.5, 0.1, 0.02], 'color': [0.3, 0.3, 0.3], 'contact': 1, 'logical': {}, 'friction': 0.1, 'X': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0]}
table threshold:  0.655
camera height:  [0.   0.9  1.22]
AAAAAAAAAAAAAAAAAAAAA:  (750000,)
(1925, 3)
(1925, 3)
[0, 0, 255]
[1.0, 0.0, 0.0]
-- WARNING:kin.cpp:addFrame:193(-1) frame already exists! returning existing without modifications!
{'name': 'table', 'ID': 1, 'rel': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0], 'shape': 'ssBox', 'size': [2.5, 2.5, 0.1, 0.02], 'color': [0.3, 0.3, 0.3], 'contact': 1, 'logical': {}, 'friction': 0.1, 'X': [0.0, 0.0, 0.6, 1.0, 0.0, 0.0, 0.0]}
table threshold:  0.655
camera height:  [0.   0.9  1.22]
AAAAAAAAAAAAAAAAAAAAA:  (750000,)
(2518, 3)
(2518, 3)
[0, 0, 255]
[1.0, 0.0, 0.0]
-- WARNING:kin.cpp:addFrame:193(-1) frame already exists! returning existing without modifications!
{'name': 'table', 'ID': 1, 'rel': [0.0, 0.0, 0.6, 1.0,

In [10]:
import numpy as np
import open3d as o3d
from scipy.optimize import linear_sum_assignment
from scipy.spatial import cKDTree


# ============================================================
# GEOMETRY-ONLY MATCHING
# No color usage.
# Uses:
#   1. yaw search
#   2. trimmed Chamfer
#   3. extent/aspect cost
#   4. top-view 2D occupancy IoU cost
# ============================================================

def to_o3d_safe(pcd_or_array):
    if isinstance(pcd_or_array, o3d.geometry.PointCloud):
        return o3d.geometry.PointCloud(pcd_or_array)

    arr = np.asarray(pcd_or_array)

    if arr.ndim == 2 and arr.shape[1] >= 3:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(arr[:, :3])
        return pcd

    raise TypeError(
        f"Cannot convert to Open3D point cloud: "
        f"type={type(pcd_or_array)}, shape={arr.shape}"
    )


def downsample_safe(pcd_or_array, voxel_size=0.006):
    pcd = to_o3d_safe(pcd_or_array)

    if voxel_size is not None and voxel_size > 0:
        pcd = pcd.voxel_down_sample(voxel_size)

    return pcd


def get_points(pcd_or_array, voxel_size=0.006):
    pcd = downsample_safe(pcd_or_array, voxel_size=voxel_size)
    pts = np.asarray(pcd.points)

    if len(pts) == 0:
        raise ValueError("Empty point cloud.")

    return pts


def yaw_rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)

    return np.array([
        [c, -s, 0.0],
        [s,  c, 0.0],
        [0.0, 0.0, 1.0],
    ])


def bbox_extent(pts):
    pts = np.asarray(pts, dtype=float)
    return pts.max(axis=0) - pts.min(axis=0)


def sorted_extent(pts):
    return np.sort(bbox_extent(pts))


def align_points_aabb_bottom(pts):
    """
    Align point cloud using:
      x/y: AABB center
      z: bottom z

    More stable than mean-centering for partial real clouds.
    """
    pts = np.asarray(pts, dtype=float)

    p_min = pts.min(axis=0)
    p_max = pts.max(axis=0)

    center_xy = (p_min[:2] + p_max[:2]) / 2.0
    bottom_z = p_min[2]

    anchor = np.array([center_xy[0], center_xy[1], bottom_z])
    aligned = pts - anchor

    return aligned, anchor


def trimmed_symmetric_chamfer(a, b, trim_ratio=0.80):
    """
    Robust Chamfer distance.
    Uses only the closest trim_ratio of distances.
    This helps with partial real point clouds.
    """
    tree_b = cKDTree(b)
    d_ab, _ = tree_b.query(a, k=1)

    tree_a = cKDTree(a)
    d_ba, _ = tree_a.query(b, k=1)

    d_ab = np.sort(d_ab)
    d_ba = np.sort(d_ba)

    n_ab = max(1, int(len(d_ab) * trim_ratio))
    n_ba = max(1, int(len(d_ba) * trim_ratio))

    return np.mean(d_ab[:n_ab]) + np.mean(d_ba[:n_ba])


def relative_extent_cost(real_pts, sim_pts):
    er = sorted_extent(real_pts)
    es = sorted_extent(sim_pts)

    return np.linalg.norm((er - es) / (es + 1e-6))


def aspect_ratio_cost(real_pts, sim_pts):
    er = sorted_extent(real_pts)
    es = sorted_extent(sim_pts)

    er_ratio = er / (er[-1] + 1e-6)
    es_ratio = es / (es[-1] + 1e-6)

    return np.linalg.norm(er_ratio - es_ratio)


def make_xy_occupancy(pts, grid_size=48):
    """
    Creates a normalized 2D top-view occupancy grid.
    This is useful for distinguishing hammer vs L-ruler.
    """
    xy = np.asarray(pts[:, :2], dtype=float)

    xy_min = xy.min(axis=0)
    xy_max = xy.max(axis=0)

    scale = np.max(xy_max - xy_min) + 1e-8

    xy_norm = (xy - xy_min) / scale
    xy_norm = np.clip(xy_norm, 0.0, 0.999)

    grid = np.zeros((grid_size, grid_size), dtype=bool)

    ij = (xy_norm * grid_size).astype(int)
    grid[ij[:, 0], ij[:, 1]] = True

    return grid


def occupancy_iou_cost(real_pts, sim_pts, grid_size=48):
    """
    2D shape mismatch cost.
    Lower is better.
    """
    g_real = make_xy_occupancy(real_pts, grid_size=grid_size)
    g_sim = make_xy_occupancy(sim_pts, grid_size=grid_size)

    inter = np.logical_and(g_real, g_sim).sum()
    union = np.logical_or(g_real, g_sim).sum()

    if union == 0:
        return 1.0

    iou = inter / union
    return 1.0 - iou


def get_world_center_from_points(pts):
    pts = np.asarray(pts, dtype=float)

    p_min = pts.min(axis=0)
    p_max = pts.max(axis=0)

    return (p_min + p_max) / 2.0


def object_geometry_gate(real_pts, obj_name):
    """
    Geometry-only gate.
    No color.

    Works for lifting and minigolf.
    """
    extent = bbox_extent(real_pts)
    sorted_e = np.sort(extent)

    small, mid, large = sorted_e
    name = str(obj_name).lower()

    # Any task platform should be horizontally large.
    if "platform" in name:
        if "minigolf" in name:
            return large > 0.20 and mid > 0.08
        elif "lifting" in name:
            return large > 0.25 and mid > 0.08
        else:
            return large > 0.20

    # Tools should not be huge platform-sized clusters.
    if name in ["hammer", "spatula", "l-ruler", "l_ruler"] or "ruler" in name:
        if large > 0.55:
            return False
        if mid > 0.35 and large > 0.40:
            return False

    return True


def best_yaw_alignment_cost(real_pts, sim_pts, n_angles=72):
    """
    AABB-bottom align both clouds.
    Rotate simulation around z.
    Compute best geometry-only score.
    """
    real_aligned, real_anchor = align_points_aabb_bottom(real_pts)
    sim_aligned, sim_anchor = align_points_aabb_bottom(sim_pts)

    real_diag = np.linalg.norm(bbox_extent(real_aligned)) + 1e-6

    best = {
        "cost": np.inf,
        "yaw": None,
        "aligned_sim": None,
        "real_centered": real_aligned,
        "sim_centered": sim_aligned,
        "real_anchor": real_anchor,
        "sim_anchor": sim_anchor,
        "chamfer": None,
        "extent": None,
        "aspect": None,
        "occupancy": None,
    }

    for theta in np.linspace(0.0, 2.0 * np.pi, n_angles, endpoint=False):
        Rz = yaw_rotation_matrix(theta)
        sim_rot = sim_aligned @ Rz.T

        chamfer = trimmed_symmetric_chamfer(
            real_aligned,
            sim_rot,
            trim_ratio=0.80
        ) / real_diag

        extent = relative_extent_cost(real_aligned, sim_rot)
        aspect = aspect_ratio_cost(real_aligned, sim_rot)
        occupancy = occupancy_iou_cost(real_aligned, sim_rot, grid_size=48)

        total = (
            6.0 * chamfer +
            2.0 * extent +
            1.5 * aspect +
            4.0 * occupancy
        )

        if total < best["cost"]:
            best["cost"] = total
            best["yaw"] = theta
            best["aligned_sim"] = sim_rot
            best["chamfer"] = chamfer
            best["extent"] = extent
            best["aspect"] = aspect
            best["occupancy"] = occupancy

    return best


def match_clusters_to_sim_geometry_only(
    clusters,
    simulation_point_clouds,
    voxel_size=0.006,
    max_allowed_cost=12.0,
    n_angles=72,
    use_gate=True,
    verbose=True,
):
    object_names = list(simulation_point_clouds.keys())

    real_pts_list = [
        get_points(cluster, voxel_size=voxel_size)
        for cluster in clusters
    ]

    sim_pts_dict = {
        name: get_points(pcd, voxel_size=voxel_size)
        for name, pcd in simulation_point_clouds.items()
    }

    cost_matrix = np.full(
        (len(real_pts_list), len(object_names)),
        1e6,
        dtype=float
    )

    details = {}

    for i, real_pts in enumerate(real_pts_list):
        for j, obj_name in enumerate(object_names):

            if use_gate and not object_geometry_gate(real_pts, obj_name):
                details[(i, obj_name)] = {
                    "rejected_by_gate": True,
                    "total": 1e6,
                }
                continue

            sim_pts = sim_pts_dict[obj_name]

            best = best_yaw_alignment_cost(
                real_pts,
                sim_pts,
                n_angles=n_angles
            )

            cost_matrix[i, j] = best["cost"]
            details[(i, obj_name)] = best

    row_ind, col_ind = linear_sum_assignment(cost_matrix)

    matches = []

    for r, c in zip(row_ind, col_ind):
        obj_name = object_names[c]
        cost = cost_matrix[r, c]

        if cost > max_allowed_cost:
            print(
                f"Rejected: cluster {r} -> {obj_name}, "
                f"cost={cost:.3f}"
            )
            continue

        real_pts = real_pts_list[r]
        sim_pts = sim_pts_dict[obj_name]
        best = details[(r, obj_name)]

        world_center = get_world_center_from_points(real_pts)

        matches.append({
            "cluster_id": r,
            "object_name": obj_name,
            "tool_name": obj_name,

            "cost": float(cost),
            "yaw_alignment": float(best["yaw"]),

            "cluster_pcd": clusters[r],
            "sim_pcd": simulation_point_clouds[obj_name],

            "real_centered": best["real_centered"],
            "aligned_sim_centered": best["aligned_sim"],

            "real_anchor": best["real_anchor"],
            "sim_anchor": best["sim_anchor"],

            "real_extent": bbox_extent(real_pts),
            "sim_extent": bbox_extent(sim_pts),

            "center": world_center,
            "position": world_center,

            "details": {
                "chamfer": best["chamfer"],
                "extent": best["extent"],
                "aspect": best["aspect"],
                "occupancy": best["occupancy"],
                "total": best["cost"],
            }
        })

    if verbose:
        print("\n================ REAL CLUSTERS ================")

        for i, pts in enumerate(real_pts_list):
            print(
                f"Cluster {i}: "
                f"points={len(pts)}, "
                f"center={get_world_center_from_points(pts).round(3)}, "
                f"extent={bbox_extent(pts).round(3)}, "
                f"sorted_extent={sorted_extent(pts).round(3)}"
            )

        print("\n================ SIM OBJECTS ================")

        for name, pts in sim_pts_dict.items():
            print(
                f"{name}: "
                f"points={len(pts)}, "
                f"extent={bbox_extent(pts).round(3)}, "
                f"sorted_extent={sorted_extent(pts).round(3)}"
            )

        print("\n================ COST MATRIX ================")
        print("Rows: real cluster ids")
        print("Cols:", object_names)
        print(np.round(cost_matrix, 3))

        print("\n================ ACCEPTED MATCHES ================")

        for m in matches:
            print(
                f"Cluster {m['cluster_id']} -> {m['object_name']} | "
                f"cost={m['cost']:.3f} | "
                f"yaw_align={m['yaw_alignment']:.3f} | "
                f"real_extent={m['real_extent'].round(3)} | "
                f"sim_extent={m['sim_extent'].round(3)} | "
                f"center={m['center'].round(3)}"
            )
            print(
                "   details:",
                {k: round(float(v), 4) for k, v in m["details"].items()}
            )

    return matches, cost_matrix, details


def pcd_from_points(pts, color):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(pts)
    pcd.paint_uniform_color(color)
    return pcd


def visualize_one_geometry_match(match):
    real = pcd_from_points(
        match["real_centered"],
        [1.0, 0.0, 0.0]
    )

    sim = pcd_from_points(
        match["aligned_sim_centered"],
        [0.0, 0.0, 1.0]
    )

    real_bbox = real.get_axis_aligned_bounding_box()
    real_bbox.color = [1.0, 0.0, 0.0]

    sim_bbox = sim.get_axis_aligned_bounding_box()
    sim_bbox.color = [0.0, 0.0, 1.0]

    print("\nVisualizing aligned geometry match:")
    print(f"  object      : {match['object_name']}")
    print(f"  cluster id  : {match['cluster_id']}")
    print(f"  cost        : {match['cost']:.3f}")
    print(f"  yaw align   : {match['yaw_alignment']:.3f}")
    print(f"  details     : {match['details']}")
    print("  red  = real cluster")
    print("  blue = simulation object after yaw search")

    o3d.visualization.draw_geometries(
        [real, sim, real_bbox, sim_bbox],
        window_name=(
            f"GEOMETRY MATCH | "
            f"{match['object_name']} | cluster {match['cluster_id']}"
        )
    )


def visualize_all_geometry_matches(matches):
    for match in matches:
        visualize_one_geometry_match(match)


# ============================================================
# RUN
# ============================================================

matches, cost_matrix, details = match_clusters_to_sim_geometry_only(
    clusters=clusters,
    simulation_point_clouds=simulation_point_clouds,
    voxel_size=0.006,
    max_allowed_cost=12.0,
    n_angles=72,
    use_gate=True,
    verbose=True,
)

visualize_all_geometry_matches(matches)


================ REAL CLUSTERS ================
Cluster 0: points=3971, center=[ 0.215 -0.679  0.177], extent=[0.221 0.282 0.35 ], sorted_extent=[0.221 0.282 0.35 ]
Cluster 1: points=282, center=[ 0.032 -0.656  0.013], extent=[0.094 0.212 0.02 ], sorted_extent=[0.02  0.094 0.212]
Cluster 2: points=365, center=[-0.145 -0.569  0.013], extent=[0.167 0.172 0.021], sorted_extent=[0.021 0.167 0.172]
Cluster 3: points=280, center=[-0.286 -0.71   0.011], extent=[0.125 0.203 0.017], sorted_extent=[0.017 0.125 0.203]

================ SIM OBJECTS ================
spatula: points=579, extent=[0.082 0.254 0.019], sorted_extent=[0.019 0.082 0.254]
L-ruler: points=763, extent=[0.156 0.219 0.017], sorted_extent=[0.017 0.156 0.219]
hammer: points=773, extent=[0.072 0.265 0.028], sorted_extent=[0.028 0.072 0.265]
hammering-platform: points=7275, extent=[0.203 0.15  0.332], sorted_extent=[0.15  0.203 0.332]

================ COST MATRIX ================
Rows: real cluster ids
Cols: ['spatula', 'L-ruler

In [ ]:
import numpy as np
import robotic as ry
import WayTu_RAI.model_utils as mutils


# ============================================================
# CREATE SHIFTED RAI ENVIRONMENT FROM MATCHES
# ============================================================

def quat_from_yaw(yaw):
    """
    RAI quaternion format: [w, x, y, z]
    yaw is rotation around z.
    """
    return [
        float(np.cos(yaw / 2.0)),
        0.0,
        0.0,
        float(np.sin(yaw / 2.0)),
    ]


def get_match(matches, name):
    for m in matches:
        if m.get("object_name", None) == name or m.get("tool_name", None) == name:
            return m
    return None


def compute_scene_offset_from_matches(
    matches,
    desired_scene_center=np.array([0.0, 0.45, 0.0]),
):
    """
    Computes a global xy shift so the reconstructed real scene is placed
    in front of the robot.
    """

    centers = []

    for m in matches:
        if "center" in m:
            centers.append(np.asarray(m["center"], dtype=float))

    if len(centers) == 0:
        raise ValueError("No centers found in matches. Cannot compute scene offset.")

    centers = np.asarray(centers)
    current_scene_center = centers.mean(axis=0)

    offset = np.zeros(3)
    offset[:2] = desired_scene_center[:2] - current_scene_center[:2]
    offset[2] = 0.0

    print("Current matched scene center:", current_scene_center.round(3))
    print("Desired scene center:", desired_scene_center.round(3))
    print("Applied scene offset:", offset.round(3))

    return offset


def get_match_position(match, z_override=None, scene_offset=None):
    pos = np.asarray(match["center"], dtype=float).copy()

    if scene_offset is not None:
        pos = pos + scene_offset

    if z_override is not None:
        pos[2] = z_override

    return pos.tolist()


def get_match_yaw(match, extra_yaw=0.0):
    yaw = float(match.get("yaw_alignment", 0.0))
    return yaw + extra_yaw


# ============================================================
# ADD TOOLS
# ============================================================

def add_real_tool_from_match(
    C,
    match,
    parent="table",
    z_override=0.055,
    extra_yaw=0.0,
    scene_offset=None,
):
    name = match["object_name"]

    pos = get_match_position(
        match,
        z_override=z_override,
        scene_offset=scene_offset,
    )

    yaw = get_match_yaw(match, extra_yaw=extra_yaw)
    quat = quat_from_yaw(yaw)

    if name == "hammer":
        handle_shape = [0.035, 0.25, 0.035, 0.008]
        head_shape = [0.07, 0.04, 0.04, 0.010]

        mutils.add_shape(
            C=C,
            frame_name="hammer",
            parent=parent,
            shape=handle_shape,
            color=[0.6, 0.25, 0.1],
            relative_position=pos,
            relative_quaternion=quat,
            joint=True,
            mass=1.0,
        )

        mutils.add_shape(
            C=C,
            frame_name="hammer-head",
            parent="hammer",
            shape=head_shape,
            color=[0.2, 0.2, 0.2],
            relative_position=[0.0, 0.14, 0.0],
            mass=0.5,
        )

    elif name == "spatula":
        handle_shape = [0.025, 0.25, 0.025, 0.005]
        head_shape = [0.08, 0.08, 0.01, 0.005]

        mutils.add_shape(
            C=C,
            frame_name="spatula",
            parent=parent,
            shape=handle_shape,
            color=[0.1, 0.7, 0.9],
            relative_position=pos,
            relative_quaternion=quat,
            joint=True,
            mass=1.0,
        )

        mutils.add_shape(
            C=C,
            frame_name="spatula-head",
            parent="spatula",
            shape=head_shape,
            color=[0.1, 0.7, 0.9],
            relative_position=[0.0, 0.15, 0.0],
            mass=0.5,
        )

    elif name == "L-ruler":
        long_shape = [0.04, 0.20, 0.02, 0.005]
        short_shape = [0.15, 0.04, 0.02, 0.005]

        mutils.add_shape(
            C=C,
            frame_name="L-ruler",
            parent=parent,
            shape=long_shape,
            color=[0.7, 0.7, 0.2],
            relative_position=pos,
            relative_quaternion=quat,
            joint=True,
            mass=1.0,
        )

        mutils.add_shape(
            C=C,
            frame_name="L-ruler-short",
            parent="L-ruler",
            shape=short_shape,
            color=[0.7, 0.7, 0.2],
            relative_position=[0.06, 0.11, 0.0],
            mass=0.5,
        )

    else:
        print(f"Skipping unknown tool: {name}")
        return None

    print(f"Added tool: {name} at {np.round(pos, 3)} yaw={yaw:.3f}")
    return name


# ============================================================
# ADD LIFTING ENVIRONMENT
# ============================================================

def add_lifting_platform_from_match(
    C,
    match,
    parent="table",
    z_override=0.04,
    extra_yaw=0.0,
    scene_offset=None,
):
    pos = get_match_position(
        match,
        z_override=z_override,
        scene_offset=scene_offset,
    )

    yaw = get_match_yaw(match, extra_yaw=extra_yaw)
    quat = quat_from_yaw(yaw)

    plate_shape = [0.2, 0.4, 0.04, 0.005]
    leg_shape = [0.04, 0.04, 0.4, 0.005]
    joint_shape = [0.06, 0.04, 0.04, 0.005]
    obj_shape = [0.03, 0.5, 0.03, 0.005]

    joint_base = [0.04, 0.0, 0.1]
    random_height = 0.0

    mutils.add_shape(
        C=C,
        frame_name="lifting-platform",
        parent=parent,
        shape=plate_shape,
        color=[0.1],
        relative_position=pos,
        relative_quaternion=quat,
        joint=True,
        mass=50.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="back-leg1",
        parent="lifting-platform",
        shape=leg_shape,
        color=[0.1],
        relative_position=[-0.04, -0.2, 0.20],
        mass=5.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="back-joint",
        parent="back-leg1",
        shape=joint_shape,
        color=[0.1],
        relative_position=joint_base,
        mass=5.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="back-leg2",
        parent="back-joint",
        shape=leg_shape,
        color=[0.1],
        relative_position=[0.04, 0.0, -0.1 - random_height],
        mass=5.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="front-leg1",
        parent="lifting-platform",
        shape=leg_shape,
        color=[0.1],
        relative_position=[-0.04, 0.2, 0.20],
        mass=5.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="front-joint",
        parent="front-leg1",
        shape=joint_shape,
        color=[0.1],
        relative_position=joint_base,
        mass=5.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="front-leg2",
        parent="front-joint",
        shape=leg_shape,
        color=[0.1],
        relative_position=[0.04, 0.0, -0.1 - random_height],
        mass=5.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="lifting-obj",
        parent="back-joint",
        shape=obj_shape,
        color=[0.0, 0.0, 1.0],
        relative_position=[0.0, 0.2, 0.04],
        joint=True,
        mass=0.00001,
    )

    print(f"Added lifting platform at {np.round(pos, 3)} yaw={yaw:.3f}")
    return "lifting-platform"


# ============================================================
# ADD MINIGOLF ENVIRONMENT
# ============================================================

def add_minigolf_platform_from_match(
    C,
    match,
    parent="table",
    z_override=0.01,
    extra_yaw=0.0,
    scene_offset=None,
):
    pos = get_match_position(
        match,
        z_override=z_override,
        scene_offset=scene_offset,
    )

    yaw = get_match_yaw(match, extra_yaw=extra_yaw)
    quat = quat_from_yaw(yaw)

    minigolf_platform_shape = [0.26, 0.4, 0.01, 0.005]
    main_area_shape = [0.26, 0.2, 0.1, 0.005]
    left_area_shape = [0.06, 0.1, 0.1, 0.005]
    right_area_shape = [0.06, 0.1, 0.1, 0.005]
    end_area_shape = [0.26, 0.1, 0.1, 0.005]
    minigolf_obj_shape = [0.05, 0.05, 0.05, 0.02]

    mutils.add_shape(
        C=C,
        frame_name="minigolf-platform",
        parent=parent,
        shape=minigolf_platform_shape,
        color=[1.0, 1.0, 1.0],
        relative_position=pos,
        relative_quaternion=quat,
        joint=True,
        mass=50.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="main-area",
        parent="minigolf-platform",
        shape=main_area_shape,
        color=[0.1],
        relative_position=[0.0, 0.1, 0.0505],
        mass=40.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="left-area",
        parent="main-area",
        shape=left_area_shape,
        color=[0.1],
        relative_position=[0.1, -0.15, 0.0],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="right-area",
        parent="main-area",
        shape=right_area_shape,
        color=[0.1],
        relative_position=[-0.1, -0.15, 0.0],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="end-area",
        parent="main-area",
        shape=end_area_shape,
        color=[0.1],
        relative_position=[0.0, -0.25, 0.0],
        mass=30.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="minigolf-obj",
        parent="minigolf-platform",
        shape=minigolf_obj_shape,
        color=[0.0, 0.0, 1.0],
        relative_position=[0.0, 0.1, 0.085],
        joint=True,
        mass=0.00001,
    )

    print(f"Added minigolf platform at {np.round(pos, 3)} yaw={yaw:.3f}")
    return "minigolf-platform"


# ============================================================
# ADD HAMMERING ENVIRONMENT
# ============================================================

def add_hammering_platform_from_match(
    C,
    match,
    parent="table",
    z_override=0.01,
    extra_yaw=0.0,
    scene_offset=None,
):
    pos = get_match_position(
        match,
        z_override=z_override,
        scene_offset=scene_offset,
    )

    yaw = get_match_yaw(match, extra_yaw=extra_yaw)
    quat = quat_from_yaw(yaw)

    hammering_platform_shape = [0.20, 0.25, 0.01, 0.001]
    main_area_shape = [0.20, 0.15, 0.25, 0.001]
    side_area_shape = [0.09, 0.15, 0.02, 0.001]
    top_area_shape = [0.20, 0.15, 0.1, 0.001]

    nail_base_shape = [0.018, 0.15, 0.018, 0.001]
    nail_head_shape = [0.04, 0.01, 0.04, 0.001]

    mutils.add_shape(
        C=C,
        frame_name="hammering-platform",
        parent=parent,
        shape=hammering_platform_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=pos,
        relative_quaternion=quat,
        joint=True,
        mass=1000.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="main-area",
        parent="hammering-platform",
        shape=main_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[0.0, -0.05, 0.13],
        mass=40.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="left-area",
        parent="main-area",
        shape=side_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[-0.055, 0.0, 0.135],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="right-area",
        parent="main-area",
        shape=side_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[0.055, 0.0, 0.135],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="top-area",
        parent="main-area",
        shape=top_area_shape,
        color=(0.76, 0.60, 0.42),
        relative_position=[0.0, 0.0, 0.195],
        mass=20.0,
    )

    mutils.add_shape(
        C=C,
        frame_name="nail-base",
        parent="hammering-platform",
        shape=nail_base_shape,
        color=[0.3, 0.5, 0.8],
        joint=True,
        mass=0.001,
    )

    mutils.add_shape(
        C=C,
        frame_name="hammering-obj",
        parent="nail-base",
        shape=nail_head_shape,
        color=[0.3, 0.5, 0.8],
        relative_position=[0.0, 0.08, 0.0],
        mass=0.001,
    )

    obj = C.getFrame("nail-base")
    platform = C.getFrame("hammering-platform").getPosition()
    obj.setPosition(platform + [0.0, 0.0, 0.265])

    print(f"Added hammering platform at {np.round(pos, 3)} yaw={yaw:.3f}")
    return "hammering-platform"


# ============================================================
# MAIN CREATE FUNCTION
# ============================================================

def create_environment_from_matches(
    matches,
    scenario_path="./rai-robotModels/scenarios/pandaSingle.g",
    task="hammering",
    parent="table",
    desired_scene_center=np.array([0.0, 0.45, 0.0]),
    view=True,
):
    """
    Creates a shifted RAI scene from matches.

    task:
        "lifting", "minigolf", or "hammering"
    """

    C = ry.Config()
    C.addFile(scenario_path)

    placed_objects = {}

    scene_offset = compute_scene_offset_from_matches(
        matches,
        desired_scene_center=desired_scene_center,
    )

    if task == "lifting":
        platform_match = get_match(matches, "lifting-platform")

        if platform_match is not None:
            placed_objects["lifting-platform"] = add_lifting_platform_from_match(
                C,
                platform_match,
                parent=parent,
                z_override=0.04,
                extra_yaw=0.0,
                scene_offset=scene_offset,
            )
        else:
            print("WARNING: lifting-platform match not found.")

    elif task == "minigolf":
        platform_match = get_match(matches, "minigolf-platform")

        if platform_match is not None:
            placed_objects["minigolf-platform"] = add_minigolf_platform_from_match(
                C,
                platform_match,
                parent=parent,
                z_override=0.01,
                extra_yaw=0.0,
                scene_offset=scene_offset,
            )
        else:
            print("WARNING: minigolf-platform match not found.")

    elif task == "hammering":
        platform_match = get_match(matches, "hammering-platform")

        if platform_match is not None:
            placed_objects["hammering-platform"] = add_hammering_platform_from_match(
                C,
                platform_match,
                parent=parent,
                z_override=0.01,
                extra_yaw=0.0,
                scene_offset=scene_offset,
            )
        else:
            print("WARNING: hammering-platform match not found.")

    else:
        raise ValueError("task must be 'lifting', 'minigolf', or 'hammering'.")

    for tool_name in ["hammer", "spatula", "L-ruler"]:
        tool_match = get_match(matches, tool_name)

        if tool_match is not None:
            placed_objects[tool_name] = add_real_tool_from_match(
                C,
                tool_match,
                parent=parent,
                z_override=0.055,
                extra_yaw=0.0,
                scene_offset=scene_offset,
            )

    if view:
        C.view()

    return C, placed_objects, scene_offset


# ============================================================
# RUN
# ============================================================

TASK = "hammering"

C_scene, placed_objects, scene_offset = create_environment_from_matches(
    matches=matches,
    scenario_path="./rai-robotModels/scenarios/pandaSingle.g",
    task=TASK,
    parent="table",

    # This controls where the whole reconstructed scene appears.
    desired_scene_center=np.array([0.0, 0.20, 0.0]),

    view=True,
)

print("Placed objects:", placed_objects)
print("Scene offset:", scene_offset)

Current matched scene center: [-0.046 -0.653  0.053]
Desired scene center: [0.  0.2 0. ]
Applied scene offset: [0.046 0.853 0.   ]
Added hammering platform at [0.261 0.175 0.01 ] yaw=0.436
Added tool: hammer at [0.078 0.198 0.055] yaw=2.880
Added tool: spatula at [-0.099  0.285  0.055] yaw=3.927
Added tool: L-ruler at [-0.24   0.143  0.055] yaw=0.698
Placed objects: {'hammering-platform': 'hammering-platform', 'hammer': 'hammer', 'spatula': 'spatula', 'L-ruler': 'L-ruler'}
Scene offset: [0.04579557 0.85330701 0.        ]


In [12]:
parameters = { 
    "mode" : "test", 
    "num-tools" : 3,
    "num-obj-points": 512,
    "label-list-all" : ["lifting-platform", "minigolf-platform", "hammering-platform",
                  "hammer", "spatula", "L-ruler", "screwdriver", "ball", "book", "thin-stick", "ring"],
    # "model-name" :  "waytu_unified_model_lifting_v23_best.pth",
    "model-name" :  "waytu_unified_model_hammering_v25_best.pth",
    # "model-name"    : "waytu_unified_model_minigolf_v22_best.pth",
    "feature-extractor-path" : "small-pointner-encoder-distractor_best.pth",
    "feature-size" : 128,
    "num-trials" : 10 , 
    "tool-type" : "primitive",
    "task" : "hammering", 
    "dataset-save-path" : "lifting-test-workshop",
    "area-middle": {
        "min" : [-0.35 , 0.15, 0.060],
        "max" : [ 0.35 , 0.45, 0.065]
        },
    "area-negative" : {
        "min": [-0.50 , 0.15, 0.050],
        "max": [ -0.40 , 0.25, 0.060],
        },
    "area-positive" : {
        "min": [0.40 , 0.15, 0.050],
        "max": [ 0.50 , 0.25, 0.060],
        },
    "cameras" : ["camera1", "camera2", "camera3" ] 

}

In [13]:
  import os
import json
import numpy as np
import torch
import robotic as ry
import WayTu_RAI.model_utils as mutils
from WayTu_RAI.generator_and_selector import WayTuUnifiedModel
from scipy.spatial.transform import Rotation as R


# ============================================================
# 1. Geometry utilities
# ============================================================

def rai_quat_to_rotmat(q_wxyz):
    """
    RAI quaternion format: [w, x, y, z]
    scipy format: [x, y, z, w]
    """
    q_wxyz = np.asarray(q_wxyz, dtype=float)
    q_xyzw = np.array([q_wxyz[1], q_wxyz[2], q_wxyz[3], q_wxyz[0]])
    return R.from_quat(q_xyzw).as_matrix()


def sample_box_surface_points(size, num_points=512):
    """
    Sample surface points from an axis-aligned ssBox in local frame.
    size = [x, y, z, radius]
    """
    sx, sy, sz = float(size[0]), float(size[1]), float(size[2])

    faces = [
        ("x",  sx / 2.0),
        ("x", -sx / 2.0),
        ("y",  sy / 2.0),
        ("y", -sy / 2.0),
        ("z",  sz / 2.0),
        ("z", -sz / 2.0),
    ]

    points = []
    per_face = max(1, num_points // len(faces))

    for axis, value in faces:
        if axis == "x":
            y = np.random.uniform(-sy / 2.0, sy / 2.0, per_face)
            z = np.random.uniform(-sz / 2.0, sz / 2.0, per_face)
            x = np.full_like(y, value)
        elif axis == "y":
            x = np.random.uniform(-sx / 2.0, sx / 2.0, per_face)
            z = np.random.uniform(-sz / 2.0, sz / 2.0, per_face)
            y = np.full_like(x, value)
        else:
            x = np.random.uniform(-sx / 2.0, sx / 2.0, per_face)
            y = np.random.uniform(-sy / 2.0, sy / 2.0, per_face)
            z = np.full_like(x, value)

        points.append(np.stack([x, y, z], axis=1))

    points = np.concatenate(points, axis=0)

    if len(points) > num_points:
        idx = np.random.choice(len(points), num_points, replace=False)
        points = points[idx]

    return points


def frame_exists(C, frame_name):
    try:
        C.getFrame(frame_name)
        return True
    except Exception:
        return False


def transform_local_points_to_world(C, frame_name, local_points):
    frame = C.getFrame(frame_name)

    pos = np.asarray(frame.getPosition(), dtype=float)
    quat = np.asarray(frame.getQuaternion(), dtype=float)

    R_world = rai_quat_to_rotmat(quat)

    return local_points @ R_world.T + pos


def canonical_scene_name_for_model(name):
    """
    Converts recreated frame/object names into model class names.
    """
    name_low = str(name).lower().replace("_", "-")
    task_name = parameters.get("task", None)

    if "spatula" in name_low:
        return "spatula"

    if "ruler" in name_low:
        return "L-ruler"

    # IMPORTANT: check hammering before hammer,
    # otherwise "hammering-platform" becomes "hammer".
    if "hammering" in name_low or "nail" in name_low:
        return "hammering-platform"

    if "hammer" in name_low:
        return "hammer"

    if "minigolf" in name_low:
        return "minigolf-platform"

    if "lifting" in name_low:
        return "lifting-platform"

    if "platform" in name_low:
        if task_name == "minigolf":
            return "minigolf-platform"
        elif task_name == "lifting":
            return "lifting-platform"
        elif task_name == "hammering":
            return "hammering-platform"

    return name


def get_target_frame_name_for_task(task_name):
    if task_name == "lifting":
        return "lifting-obj"
    elif task_name == "minigolf":
        return "minigolf-obj"
    elif task_name == "hammering":
        return "hammering-obj"
    else:
        return None


# ============================================================
# 2. Current recreated-scene frame definitions
# ============================================================

CURRENT_TOOL_PARTS = {
    "hammer": {
        "base_frame": "hammer",
        "head_frame": "hammer-head",
        "base_shape": [0.025, 0.28, 0.025, 0.005],
        "head_shape": [0.12, 0.04, 0.04, 0.005],
    },
    "spatula": {
        "base_frame": "spatula",
        "head_frame": "spatula-head",
        "base_shape": [0.025, 0.25, 0.018, 0.004],
        "head_shape": [0.09, 0.10, 0.012, 0.004],
    },
    "L-ruler": {
        "base_frame": "L-ruler",
        "head_frame": "L-ruler-short",
        "base_shape": [0.025, 0.25, 0.018, 0.004],
        "head_shape": [0.14, 0.025, 0.018, 0.004],
    },
}


CURRENT_LIFTING_PLATFORM_PARTS = {
    "lifting-platform": [0.2, 0.4, 0.04, 0.005],
    "back-leg1":        [0.04, 0.04, 0.4, 0.005],
    "back-joint":       [0.06, 0.04, 0.04, 0.005],
    "back-leg2":        [0.04, 0.04, 0.4, 0.005],
    "front-leg1":       [0.04, 0.04, 0.4, 0.005],
    "front-joint":      [0.06, 0.04, 0.04, 0.005],
    "front-leg2":       [0.04, 0.04, 0.4, 0.005],
    "lifting-obj":      [0.03, 0.5, 0.03, 0.005],
}


CURRENT_MINIGOLF_PLATFORM_PARTS = {
    "minigolf-platform": [0.26, 0.4, 0.01, 0.005],
    "main-area":         [0.26, 0.2, 0.1, 0.005],
    "left-area":         [0.06, 0.1, 0.1, 0.005],
    "right-area":        [0.06, 0.1, 0.1, 0.005],
    "end-area":          [0.26, 0.1, 0.1, 0.005],
    "minigolf-obj":      [0.05, 0.05, 0.05, 0.02],
}


CURRENT_HAMMERING_PLATFORM_PARTS = {
    "hammering-platform": [0.20, 0.25, 0.01, 0.001],
    "main-area":          [0.20, 0.15, 0.25, 0.001],
    "left-area":          [0.09, 0.15, 0.02, 0.001],
    "right-area":         [0.09, 0.15, 0.02, 0.001],
    "top-area":           [0.20, 0.15, 0.1, 0.001],
    "nail-base":          [0.018, 0.15, 0.018, 0.001],
    "hammering-obj":      [0.04, 0.01, 0.04, 0.001],
}


# ============================================================
# 3. Convert current placed_objects if needed
# ============================================================

def normalize_placed_objects_for_current_code(placed_objects):
    """
    Accepts either:
        dict: {"hammer": "hammer", ...}
    or:
        list of {"object_type": ..., "base_name": ...}

    Returns a list of dictionaries.
    """

    if isinstance(placed_objects, list):
        return placed_objects

    if isinstance(placed_objects, dict):
        normalized = []

        for object_type, base_name in placed_objects.items():
            normalized.append({
                "object_type": object_type,
                "base_name": base_name,
            })

        return normalized

    raise TypeError(f"Unsupported placed_objects type: {type(placed_objects)}")


# ============================================================
# 4. Build labeled model point cloud from current C_scene
# ============================================================

def sample_current_tool_from_scene(
    C,
    tool_name,
    label_id,
    num_points_per_part=512,
):
    tool_name = canonical_scene_name_for_model(tool_name)

    if tool_name not in CURRENT_TOOL_PARTS:
        print(f"[Skipping] Unknown tool for sampling: {tool_name}")
        return None

    spec = CURRENT_TOOL_PARTS[tool_name]

    points_all = []

    for frame_name, shape in [
        (spec["base_frame"], spec["base_shape"]),
        (spec["head_frame"], spec["head_shape"]),
    ]:
        if frame_exists(C, frame_name):
            local = sample_box_surface_points(
                shape,
                num_points=num_points_per_part
            )
            world = transform_local_points_to_world(C, frame_name, local)
            points_all.append(world)
        else:
            print(f"[Warning] Missing tool frame: {frame_name}")

    if len(points_all) == 0:
        return None

    points = np.concatenate(points_all, axis=0)
    labels = np.full((points.shape[0], 1), label_id)

    return np.concatenate([points, labels], axis=1)


def sample_platform_parts_from_scene(
    C,
    part_shapes,
    label_id,
    num_points_per_part=512,
    platform_name="platform",
):
    points_all = []

    for frame_name, shape in part_shapes.items():
        if not frame_exists(C, frame_name):
            print(f"[Warning] Missing {platform_name} frame: {frame_name}")
            continue

        local = sample_box_surface_points(
            shape,
            num_points=num_points_per_part
        )
        world = transform_local_points_to_world(C, frame_name, local)
        points_all.append(world)

    if len(points_all) == 0:
        raise ValueError(f"No {platform_name} frames were found in C_scene.")

    points = np.concatenate(points_all, axis=0)
    labels = np.full((points.shape[0], 1), label_id)

    return np.concatenate([points, labels], axis=1)


def build_model_pointcloud_from_current_scene(
    C_scene,
    placed_objects,
    parameters,
    num_points_per_part=512,
):
    """
    Creates [N, 4] point cloud:
        x, y, z, label_id
    """

    placed_objects_list = normalize_placed_objects_for_current_code(placed_objects)

    label_list = parameters["label-list-all"]
    pcl_parts = []

    print("\n================ BUILDING MODEL POINT CLOUD FROM CURRENT C_scene ================")

    for obj in placed_objects_list:
        object_type = canonical_scene_name_for_model(obj["object_type"])

        if object_type not in label_list:
            print(f"[Skipping] {object_type} is not in label-list-all.")
            continue

        label_id = label_list.index(object_type)

        if object_type in ["hammer", "spatula", "L-ruler"]:
            pcl_obj = sample_current_tool_from_scene(
                C=C_scene,
                tool_name=object_type,
                label_id=label_id,
                num_points_per_part=num_points_per_part,
            )

        elif object_type == "lifting-platform":
            pcl_obj = sample_platform_parts_from_scene(
                C=C_scene,
                part_shapes=CURRENT_LIFTING_PLATFORM_PARTS,
                label_id=label_id,
                num_points_per_part=num_points_per_part,
                platform_name="lifting-platform",
            )

        elif object_type == "minigolf-platform":
            pcl_obj = sample_platform_parts_from_scene(
                C=C_scene,
                part_shapes=CURRENT_MINIGOLF_PLATFORM_PARTS,
                label_id=label_id,
                num_points_per_part=num_points_per_part,
                platform_name="minigolf-platform",
            )

        elif object_type == "hammering-platform":
            pcl_obj = sample_platform_parts_from_scene(
                C=C_scene,
                part_shapes=CURRENT_HAMMERING_PLATFORM_PARTS,
                label_id=label_id,
                num_points_per_part=num_points_per_part,
                platform_name="hammering-platform",
            )

        else:
            print(f"[Skipping] Unknown object type: {object_type}")
            continue

        if pcl_obj is not None:
            pcl_parts.append(pcl_obj)
            print(
                f"{object_type:20s} label={label_id:2d} "
                f"points={len(pcl_obj)}"
            )

    if len(pcl_parts) == 0:
        raise ValueError("No model point cloud was created from C_scene.")

    point_clouds_labels = np.concatenate(pcl_parts, axis=0)

    print("Final model point cloud:", point_clouds_labels.shape)

    return point_clouds_labels


def uniform_sample_per_label(point_clouds_labels, num_points_per_object=512):
    sampled = []

    labels = point_clouds_labels[:, 3].astype(int)
    unique_labels = np.unique(labels)

    for label in unique_labels:
        pts = point_clouds_labels[labels == label]

        replace = len(pts) < num_points_per_object

        idx = np.random.choice(
            len(pts),
            size=num_points_per_object,
            replace=replace,
        )

        sampled.append(pts[idx])

    return np.concatenate(sampled, axis=0)


# ============================================================
# 5. Draw predicted waypoints
# ============================================================

def add_waypoint(C, frame_name, position, orientation):
    try:
        C.delFrame(frame_name)
    except Exception:
        pass

    wp = C.addFrame(frame_name)
    wp.setShape(ry.ST.marker, size=[0.1])
    wp.setQuaternion(orientation)
    wp.setPosition(position)
    return wp


def draw_predicted_waypoints_in_scene(C, best_wp, prefix="predicted"):
    positions = np.asarray(best_wp["pos"])

    if "qua" in best_wp:
        orientations = np.asarray(best_wp["qua"])
    else:
        orientations = np.tile(np.array([1.0, 0.0, 0.0, 0.0]), (3, 1))

    names = ["grasp", "initial", "goal"]

    for i in range(3):
        frame_name = f"{prefix}-{names[i]}-wp"

        position = positions[i].tolist()

        if orientations.ndim == 2:
            orientation = orientations[i].tolist()
        else:
            orientation = [1.0, 0.0, 0.0, 0.0]

        add_waypoint(
            C=C,
            frame_name=frame_name,
            position=position,
            orientation=orientation,
        )

        print(f"Waypoint {i} ({names[i]}):")
        print("  position:", position)
        print("  orientation:", orientation)


# ============================================================
# 6. Mirroring helpers
# ============================================================

def rotate_quat_yaw_180_safe(q):
    q_np = np.asarray(q, dtype=float)
    return np.asarray(mutils.rotate_quat_yaw_180(q_np), dtype=float)


def torch_set_quat_yaw_180(quat_tensor, waypoint_idx):
    q_np = quat_tensor[0, waypoint_idx].detach().cpu().numpy()
    q_rot = rotate_quat_yaw_180_safe(q_np)

    quat_tensor[0, waypoint_idx] = torch.tensor(
        q_rot,
        dtype=quat_tensor.dtype,
        device=quat_tensor.device,
    )


# ============================================================
# 7. Run WayTu model on current recreated scene
# ============================================================

def run_waytu_on_current_recreated_scene(
    C_scene,
    placed_objects,
    parameters,
    model_dir="models",
    num_points_per_part=512,
    visualize=True,
):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    task_name = parameters.get("task", "lifting")

    model_path = os.path.join(model_dir, parameters["model-name"])
    print("Loading model:", model_path)

    model = WayTuUnifiedModel(parameters, device).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    label_list = parameters["label-list-all"]

    env_label_idx = [
        i for i, name in enumerate(label_list)
        if "platform" in name
    ]

    tool_label_idx = [
        i for i, name in enumerate(label_list)
        if "platform" not in name
    ]

    point_clouds_labels = build_model_pointcloud_from_current_scene(
        C_scene=C_scene,
        placed_objects=placed_objects,
        parameters=parameters,
        num_points_per_part=num_points_per_part,
    )

    pcl = uniform_sample_per_label(
        point_clouds_labels,
        num_points_per_object=parameters["num-obj-points"],
    )

    xyz = pcl[:, :3]
    labels = pcl[:, 3].astype(int)

    env_mask = np.isin(labels, env_label_idx)
    env_pc = xyz[env_mask].copy()

    if len(env_pc) < 10:
        raise ValueError("No environment/platform points found.")

    # Original hammering mirror logic
    hammering_flag = False
    mean_x = np.mean(env_pc[:, 0])

    if mean_x < 0 and task_name == "hammering":
        print("[MIRROR] Hammering env mean_x < 0. Mirroring env_pc x.")
        env_pc[:, 0] = -env_pc[:, 0]
        hammering_flag = True

    env_center = env_pc.mean(axis=0)
    env_pc_norm = env_pc - env_center
    env_scale = np.linalg.norm(env_pc_norm, axis=1).max()
    env_pc_norm = env_pc_norm / (env_scale + 1e-8)

    env_labels = labels[env_mask]
    platform_label = np.bincount(env_labels).argmax()
    platform_class_idx = env_label_idx.index(platform_label)

    env_onehot = np.zeros(len(env_label_idx), dtype=np.float32)
    env_onehot[platform_class_idx] = 1.0

    env_tensor = torch.tensor(
        env_pc_norm,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    env_onehot_tensor = torch.tensor(
        env_onehot,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    target_frame_name = get_target_frame_name_for_task(task_name)

    if target_frame_name is not None and frame_exists(C_scene, target_frame_name):
        target_center_np = np.asarray(
            C_scene.getFrame(target_frame_name).getPosition(),
            dtype=np.float32,
        )
        print(f"Using target center from frame: {target_frame_name}", target_center_np)
    else:
        target_center_np = env_pc.mean(axis=0).astype(np.float32)
        print("[Warning] Target frame not found. Using env_pc mean as target center.")

    print("**DEBUG** target_center old:", target_center_np)
    print("**DEBUG** env_center old:", env_center)

    # Original lifting side fix
    lifting_side_fix = False

    if target_center_np[0] < 0 and task_name == "lifting":
        lifting_side_fix = True
        print("[MIRROR] Lifting target x < 0. Applying lifting side fix.")

    if lifting_side_fix:
        target_center_np[0] = -target_center_np[0]
        env_center[0] = -env_center[0]

    print("**DEBUG** target_center new:", target_center_np)
    print("**DEBUG** env_center new:", env_center)

    target_center = torch.tensor(
        target_center_np,
        dtype=torch.float32
    ).unsqueeze(0).to(device)

    tool_scores = {}
    tool_waypoints = {}

    print("\n================ MODEL PREDICTIONS ON CURRENT RECREATED SCENE ================")

    for tool_idx in tool_label_idx:
        tool_name = label_list[tool_idx]

        tool_points = xyz[labels == tool_idx]

        if len(tool_points) < 10:
            print(f"[Skipping] Not enough points for tool: {tool_name}")
            continue

        tool_center = tool_points.mean(axis=0)
        tool_points_norm = tool_points - tool_center
        tool_scale = np.linalg.norm(tool_points_norm, axis=1).max()
        tool_points_norm = tool_points_norm / (tool_scale + 1e-8)

        tool_tensor = torch.tensor(
            tool_points_norm,
            dtype=torch.float32
        ).unsqueeze(0).to(device)

        params = {
            "tool_centers": torch.tensor(tool_center, dtype=torch.float32).unsqueeze(0).to(device),
            "tool_scales": torch.tensor(tool_scale, dtype=torch.float32).unsqueeze(0).to(device),
            "env_centers": torch.tensor(env_center, dtype=torch.float32).unsqueeze(0).to(device),
            "env_scales": torch.tensor(env_scale, dtype=torch.float32).unsqueeze(0).to(device),
            "target_center": target_center,
        }

        with torch.no_grad():
            pos, quat, score, yaw = model(
                tool_tensor,
                env_tensor,
                env_onehot_tensor,
                params,
            )

        tool_center_t = params["tool_centers"][0]
        tool_scale_t = params["tool_scales"][0]
        env_center_t = params["env_centers"][0]
        env_scale_t = params["env_scales"][0]

        tool_pos = pos[0, 0] * tool_scale_t + tool_center_t
        env_pos1 = pos[0, 1] * env_scale_t + env_center_t
        env_pos2 = pos[0, 2] * env_scale_t + env_center_t

        # Mirror back after prediction
        if lifting_side_fix:
            print(f"[MIRROR BACK] lifting for {tool_name}")
            env_pos1[0] = -env_pos1[0]
            env_pos2[0] = -env_pos2[0]
            torch_set_quat_yaw_180(quat, waypoint_idx=1)
            torch_set_quat_yaw_180(quat, waypoint_idx=2)

        if hammering_flag:
            print(f"[MIRROR BACK] hammering for {tool_name}")
            env_pos1[0] = -env_pos1[0]
            env_pos2[0] = -env_pos2[0]
            torch_set_quat_yaw_180(quat, waypoint_idx=1)
            torch_set_quat_yaw_180(quat, waypoint_idx=2)

        pos_denorm = torch.stack(
            [tool_pos, env_pos1, env_pos2],
            dim=0,
        )

        tool_scores[tool_name] = float(score.item())

        tool_waypoints[tool_name] = {
            "pos": pos_denorm.detach().cpu().numpy(),
            "qua": quat.squeeze(0).detach().cpu().numpy(),
            "yaw": yaw.squeeze(0).detach().cpu().numpy(),
        }

        print(f"\nTool: {tool_name}")
        print("score:", round(tool_scores[tool_name], 4))
        print("positions:")
        print(tool_waypoints[tool_name]["pos"])
        print("quaternions:")
        print(tool_waypoints[tool_name]["qua"])
        print("yaw:")
        print(tool_waypoints[tool_name]["yaw"])

    if len(tool_scores) == 0:
        raise ValueError("No tool predictions were produced.")

    best_tool = max(tool_scores, key=tool_scores.get)
    best_score = tool_scores[best_tool]
    best_wp = tool_waypoints[best_tool]

    print("\n================ SELECTED TOOL ================")
    print("Best tool:", best_tool)
    print("Best score:", best_score)
    print("Best positions:")
    print(best_wp["pos"])
    print("Best quaternions:")
    print(best_wp["qua"])
    print("Best yaw:")
    print(best_wp["yaw"])

    if visualize:
        draw_predicted_waypoints_in_scene(
            C_scene,
            best_wp,
            prefix=f"predicted-{best_tool}",
        )
        C_scene.view()

    return best_tool, best_score, best_wp, tool_scores, tool_waypoints, point_clouds_labels


# ============================================================
# 8. Save outputs
# ============================================================

def save_waytu_outputs_to_json(
    best_tool,
    best_score,
    best_wp,
    tool_scores,
    tool_waypoints,
    filename="real_robot_waypoints.json",
):
    waypoint_data = {
        "best_tool": str(best_tool),
        "best_score": float(best_score),
        "best_waypoints": {
            "positions": np.asarray(best_wp["pos"], dtype=float).tolist(),
            "quaternions": np.asarray(best_wp["qua"], dtype=float).tolist(),
            "yaw": float(np.asarray(best_wp["yaw"]).reshape(-1)[0]),
        },
        "all_tools": [],
    }

    for tool_name, wp in tool_waypoints.items():
        waypoint_data["all_tools"].append({
            "tool": str(tool_name),
            "score": float(tool_scores[tool_name]),
            "waypoints": {
                "positions": np.asarray(wp["pos"], dtype=float).tolist(),
                "quaternions": np.asarray(wp["qua"], dtype=float).tolist(),
                "yaw": float(np.asarray(wp["yaw"]).reshape(-1)[0]),
            },
        })

    with open(filename, "w") as f:
        json.dump(waypoint_data, f, indent=4)

    print(f"Saved WayTu waypoints to {filename}")
    return waypoint_data


def save_rai_config_clean(C, filename="real_robot_scene.g"):
    g_text = C.write()

    g_text = g_text.replace(
        "/home/ece/git/WayTU-002/retired/",
        ""
    )

    cleaned_lines = []
    for line in g_text.splitlines():
        stripped = line.strip()

        if stripped.startswith("predicted-") and "-wp:" in stripped:
            continue

        cleaned_lines.append(line)

    g_text = "\n".join(cleaned_lines) + "\n"

    with open(filename, "w") as f:
        f.write(g_text)

    print(f"Saved RAI scene to {filename}")


# ============================================================
# 9. RUN
# ============================================================

# Make sure these are already set before this cell:
# parameters["task"] = "hammering"
# parameters["model-name"] = "YOUR_HAMMERING_MODEL_NAME.pth"

best_tool, best_score, best_wp, tool_scores, tool_waypoints, model_pcl = run_waytu_on_current_recreated_scene(
    C_scene=C_scene,
    placed_objects=placed_objects,
    parameters=parameters,
    model_dir="models",
    num_points_per_part=512,
    visualize=True,
)

waypoint_data = save_waytu_outputs_to_json(
    best_tool=best_tool,
    best_score=best_score,
    best_wp=best_wp,
    tool_scores=tool_scores,
    tool_waypoints=tool_waypoints,
    filename="real_robot_waypoints.json",
)

save_rai_config_clean(
    C_scene,
    filename="real_robot_scene.g",
)

Loading model: models/waytu_unified_model_hammering_v25_best.pth
Loading frozen feature extractor from models/small-pointner-encoder-distractor_best.pth


/home/ece/git/WayTU-002/retired/WayTu_RAI/generator_and_selector.py:140: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.encoder.load_state_dict(torch.load(encoder_ckpt, 


================ BUILDING MODEL POINT CLOUD FROM CURRENT C_scene ================
hammering-platform   label= 2 points=3570
hammer               label= 3 points=1020
spatula              label= 4 points=1020
L-ruler              label= 5 points=1020
Final model point cloud: (6630, 4)
Using target center from frame: hammering-obj [0.22739877 0.24711774 0.61      ]
**DEBUG** target_center old: [0.22739877 0.24711774 0.61      ]
**DEBUG** env_center old: [0.26761424 0.16010736 0.75481166]
**DEBUG** target_center new: [0.22739877 0.24711774 0.61      ]
**DEBUG** env_center new: [0.26761424 0.16010736 0.75481166]

================ MODEL PREDICTIONS ON CURRENT RECREATED SCENE ================

Tool: hammer
score: 0.688
positions:
[[0.00741805 0.12030379 0.6680128 ]
 [0.02086109 0.17863752 0.7822178 ]
 [0.05400452 0.17102084 0.78858906]]
quaternions:
[[-0.97283554 -0.00612511  0.02162725  0.23040363]
 [-0.98192793 -0.01790689  0.08670444  0.16727003]
 [ 0.98496795  0.03969605 -0.13464333 -0.

In [14]:
import json
import re
import numpy as np


# ============================================================
# SAVE WAYTU WAYPOINTS
# ============================================================

def waypoint_to_jsonable(wp):
    """
    Converts model waypoint dict to JSON format.

    Supports both:
        wp["pos"], wp["qua"]
    and:
        wp["positions"], wp["quaternions"]
    """

    if "pos" in wp:
        positions = wp["pos"]
    elif "positions" in wp:
        positions = wp["positions"]
    else:
        raise KeyError("Waypoint dictionary has no 'pos' or 'positions' key.")

    if "qua" in wp:
        quaternions = wp["qua"]
    elif "quaternions" in wp:
        quaternions = wp["quaternions"]
    else:
        raise KeyError("Waypoint dictionary has no 'qua' or 'quaternions' key.")

    data = {
        "positions": np.asarray(positions, dtype=float).tolist(),
        "quaternions": np.asarray(quaternions, dtype=float).tolist(),
    }

    if "yaw" in wp:
        yaw_arr = np.asarray(wp["yaw"], dtype=float)

        # If yaw is scalar or single-value array, save it as float.
        # If it has multiple values, save it as list.
        if yaw_arr.size == 1:
            data["yaw"] = float(yaw_arr.reshape(-1)[0])
        else:
            data["yaw"] = yaw_arr.tolist()

    return data


def save_waypoints_to_json(
    best_tool,
    best_score,
    best_wp,
    tool_scores,
    tool_waypoints,
    filename="real_robot_waypoints.json"
):
    waypoint_data = {
        "best_tool": str(best_tool),
        "best_score": float(best_score),
        "best_waypoints": waypoint_to_jsonable(best_wp),
        "all_tools": []
    }

    for tool_name, wp in tool_waypoints.items():
        waypoint_data["all_tools"].append({
            "tool": str(tool_name),
            "score": float(tool_scores[tool_name]),
            "waypoints": waypoint_to_jsonable(wp)
        })

    with open(filename, "w") as f:
        json.dump(waypoint_data, f, indent=4)

    print(f"Saved waypoints to {filename}")
    return waypoint_data


# ============================================================
# SAVE CLEAN RAI CONFIG
# ============================================================

def save_rai_config(C, filename="real_robot_scene.g"):
    """
    Save the current RAI Config as a cleaned .g file.

    Cleaning:
    1. Removes the absolute path prefix:
       /home/ece/git/WayTU-002/retired/
    2. Removes predicted waypoint marker frames from the saved file.
    """

    g_text = C.write()

    # Remove absolute path prefix
    g_text = g_text.replace(
        "/home/ece/git/WayTU-002/retired/",
        ""
    )

    cleaned_lines = []

    for line in g_text.splitlines():
        stripped = line.strip()

        # Remove predicted waypoint marker frames, for example:
        # predicted-spatula-grasp-wp: { ... }
        # predicted-spatula-initial-wp: { ... }
        # predicted-spatula-goal-wp: { ... }
        if stripped.startswith("predicted-") and "-wp:" in stripped:
            continue

        cleaned_lines.append(line)

    g_text = "\n".join(cleaned_lines) + "\n"

    with open(filename, "w") as f:
        f.write(g_text)

    print(f"Saved cleaned RAI scene to {filename}")


# ============================================================
# RUN MODEL + SAVE OUTPUTS
# ============================================================


best_tool, best_score, best_wp, tool_scores, tool_waypoints, model_pcl = run_waytu_on_current_recreated_scene(
    C_scene=C_scene,
    placed_objects=placed_objects,
    parameters=parameters,
    model_dir="models",
    num_points_per_part=512,
    visualize=True,
)

waypoint_data = save_waypoints_to_json(
    best_tool=best_tool,
    best_score=best_score,
    best_wp=best_wp,
    tool_scores=tool_scores,
    tool_waypoints=tool_waypoints,
    filename="real_robot_waypoints.json"
)

save_rai_config(
    C_scene,
    filename="real_robot_scene.g"
)

Loading model: models/waytu_unified_model_hammering_v25_best.pth
Loading frozen feature extractor from models/small-pointner-encoder-distractor_best.pth

================ BUILDING MODEL POINT CLOUD FROM CURRENT C_scene ================
hammering-platform   label= 2 points=3570
hammer               label= 3 points=1020
spatula              label= 4 points=1020
L-ruler              label= 5 points=1020
Final model point cloud: (6630, 4)
Using target center from frame: hammering-obj [0.22739877 0.24711774 0.61      ]
**DEBUG** target_center old: [0.22739877 0.24711774 0.61      ]
**DEBUG** env_center old: [0.26837414 0.15480397 0.75058977]
**DEBUG** target_center new: [0.22739877 0.24711774 0.61      ]
**DEBUG** env_center new: [0.26837414 0.15480397 0.75058977]

================ MODEL PREDICTIONS ON CURRENT RECREATED SCENE ================

Tool: hammer
score: 0.695
positions:
[[-0.00376489  0.11005923  0.6688057 ]
 [ 0.02481906  0.15411139  0.78027385]
 [ 0.05975387  0.14945118  0.78475

/tmp/ipykernel_10435/1098315764.py:493: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(model_path, map_location=device))
